### Importing Important Libraries.

In [ ]:
import pandas as pd
import os
import sqlite3
from pathlib import Path
import shutil

### Initial Findings.

In [ ]:
p_and_l = pd.read_excel(r"C:\Users\bhrra\Desktop\nifty100\data\raw\profitandloss.xlsx",skiprows=1)
doc = pd.read_excel(r"C:\Users\bhrra\Desktop\nifty100\data\raw\documents.xlsx",skiprows=1)
p_and_c = pd.read_excel(r"C:\Users\bhrra\Desktop\nifty100\data\raw\prosandcons.xlsx",skiprows=1)
companies_df = pd.read_excel(r"C:\Users\bhrra\Desktop\nifty100\data\raw\companies.xlsx",skiprows=1)
bs_df = pd.read_excel(r"C:\Users\bhrra\Desktop\nifty100\data\raw\balancesheet.xlsx",skiprows=1)
print(bs_df.head(2))
print("_"*100)
print(companies_df.head(2))
print("_"*105)
print(bs_df.columns)
print("_"*105)
print(companies_df.columns)
print(bs_df.info())
print("_"*110)
print(companies_df.info())
print(companies_df.isnull().sum())
nulls = companies_df.isnull().sum()
nulls[nulls > 0]

companies_df[companies_df.isnull().any(axis=1)]
print(p_and_l.head(5))
print('-'*100)
print(doc.head(5))
print("-"*100)
print(p_and_c.head(5))
p_and_l.columns
len(p_and_l["company_id"].unique())
len(p_and_c["company_id"].unique())
len(doc["company_id"].unique())
print(set(p_and_l["company_id"].unique()) - set(doc["company_id"].unique()).intersection(set(p_and_l["company_id"].unique())))
p_and_l[p_and_l["company_id"] == 'DIVISLAB']
# doc[doc["company_id"] == 'DIVISLAB']
a =(p_and_l["year"].unique())
print(a)
p_and_l["company_id"].unique()
doc["Year"].unique()

### Excel Loader & Normaliser
loader.py · normalize_year() · normalize_ticker() · 35+ unit tests

### Code is written in src/etl

In [ ]:
from loader import normalize_year,normalize_ticker

p_and_l['year'] = p_and_l["year"].apply(normalize_year)
p_and_l['company_id'] = p_and_l["company_id"].apply(normalize_ticker)
p_and_c['company_id'] = p_and_c["company_id"].apply(normalize_ticker)
doc['Year'] = doc["Year"].apply(normalize_year)


In [ ]:
print(p_and_l["year"].unique())
print("-"*100)
print(doc["Year"].unique())

In [ ]:
len(companies_df["id"].unique())


### Schema Validator — 16 DQ Rules
    DQ-01 … DQ-16 · CRITICAL PK/FK · WARNING OPM, balance, sales · generate validation_failures.csv

`Run Validator.py file in output folder.`

In [ ]:

BASE_PATH = r"C:\Users\bhrra\Desktop\nifty100"
SCHEMA_PATH = os.path.join(BASE_PATH, "db", "schema.sql")

BASE_PATH = r"C:\Users\bhrra\Desktop\nifty100"

DB_PATH = os.path.join(BASE_PATH, "db", "nifty100.db")
CLEANED_PATH = os.path.join(BASE_PATH, "data", "processed")



### Full Data Load — All 12 Files
- 7 core + 5 supplementary · load order · load_audit.csv · row counts: companies=92, P&L~1276, BS~1312, CF~1187, prices=5520 · FK check 0

In [ ]:
RAW_DIR = Path(r"C:\Users\bhrra\Desktop\nifty100\data\raw")

SUPP_DIR = RAW_DIR / "supporting datasets"

OUT_DIR = Path(r"C:\Users\bhrra\Desktop\nifty100\output")

OUT_DIR.mkdir(exist_ok=True)

CORE_FILES = [
    "companies", "profitandloss", "balancesheet", "cashflow",
    "analysis", "documents", "prosandcons"
]

SUPP_FILES = [
    "sectors", "stock_prices", "market_cap",
    "financial_ratios", "peer_groups"
]


audit = []


def split_nulls(name: str, path: Path, header_row: int):

    df = pd.read_excel(path, header=header_row)

    null_mask = df.isnull().any(axis=1)

    clean_df = df[~null_mask]
    null_df = df[null_mask]

    clean_df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    null_df.to_csv(OUT_DIR / f"{name}_null_values.csv", index=False)

    # Store audit information
    audit.append({
        "table_name": name,
        "number_of_rows": len(df),
        "clean_rows": len(clean_df),
        "null_rows": len(null_df)
    })

    print(
        f"{name}: {len(df)} rows -> "
        f"{len(clean_df)} clean, {len(null_df)} null"
    )


# Create audit DataFrame
audit_df = pd.DataFrame(audit)

# Save audit report
audit_df.to_csv(OUT_DIR / "load_audit.csv", index=False)


In [ ]:
RAW_DIR = Path(r"C:\Users\bhrra\Desktop\nifty100\data\raw")
SUPP_DIR = RAW_DIR / "supporting datasets"
OUT_DIR = Path(r"C:\Users\bhrra\Desktop\nifty100\output")
OUT_DIR.mkdir(exist_ok=True)

CORE_FILES = ["companies", "profitandloss", "balancesheet", "cashflow",
              "analysis", "documents", "prosandcons"]
SUPP_FILES = ["sectors", "stock_prices", "market_cap",
              "financial_ratios", "peer_groups"]

def split_nulls(name: str, path: Path, header_row: int):
    df = pd.read_excel(path, header=header_row)

    null_mask = df.isnull().any(axis=1)     # True if ANY column is null in that row
    clean_df = df[~null_mask]
    null_df = df[null_mask]

    clean_df.to_csv(OUT_DIR / f"{name}.csv", index=False)
    null_df.to_csv(OUT_DIR / f"{name}_null_values.csv", index=False)

    print(f"{name}: {len(df)} rows -> {len(clean_df)} clean, {len(null_df)} null")

for name in CORE_FILES:
    split_nulls(name, RAW_DIR / f"{name}.xlsx", header_row=1)

for name in SUPP_FILES:
    split_nulls(name, SUPP_DIR / f"{name}.xlsx", header_row=0)

### 4. SQLite Database Schema
schema.sql · 12 tables · PK/FK · loader.py · PRAGMA foreign_keys = ON

`companies, profitandloss, balancesheet, cashflow, analysis, documents, prosandcons, sectors, stock_prices, financial_ratios, peer_groups, market_cap`

In [ ]:
schema_sql = """CREATE TABLE companies (
    id                TEXT PRIMARY KEY,
    company_logo      TEXT,
    company_name      TEXT NOT NULL,
    chart_link        TEXT,
    about_company     TEXT,
    website           TEXT,
    nse_profile       TEXT,
    bse_profile       TEXT,
    face_value        REAL NOT NULL,
    book_value        REAL,
    roce_percentage   REAL,
    roe_percentage    REAL
);

CREATE TABLE profitandloss (
    company_id         TEXT NOT NULL,
    year                TEXT NOT NULL,
    sales                REAL NOT NULL,
    expenses             REAL,
    operating_profit     REAL,
    opm_percentage       REAL,
    other_income         REAL,
    interest             REAL,
    depreciation         REAL,
    profit_before_tax    REAL,
    tax_percentage       REAL,
    net_profit           REAL,
    eps                  REAL,
    dividend_payout      REAL,
    PRIMARY KEY (company_id, year),
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE balancesheet (
    company_id         TEXT NOT NULL,
    year                TEXT NOT NULL,
    equity_capital       REAL NOT NULL,
    reserves              REAL,
    borrowings            REAL,
    other_liabilities     REAL,
    total_liabilities     REAL NOT NULL,
    fixed_assets          REAL,
    cwip                  REAL,
    investments           REAL,
    other_asset           REAL,
    total_assets          REAL NOT NULL,
    PRIMARY KEY (company_id, year),
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE cashflow (
    company_id          TEXT NOT NULL,
    year                 TEXT NOT NULL,
    operating_activity    REAL,
    investing_activity    REAL,
    financing_activity    REAL,
    net_cash_flow         REAL,
    PRIMARY KEY (company_id, year),
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE analysis (
    company_id                 TEXT PRIMARY KEY,
    compounded_sales_growth    TEXT,
    compounded_profit_growth   TEXT,
    stock_price_cagr           TEXT,
    roe                        TEXT,
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE documents (
    id             INTEGER PRIMARY KEY AUTOINCREMENT,
    company_id     TEXT NOT NULL,
    year           INTEGER NOT NULL,
    annual_report  TEXT,
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE prosandcons (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    company_id  TEXT NOT NULL,
    pros        TEXT,
    cons        TEXT,
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE sectors (
    company_id           TEXT PRIMARY KEY,
    broad_sector          TEXT NOT NULL,
    sub_sector            TEXT,
    index_weight_pct      REAL,
    market_cap_category   TEXT,
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE stock_prices (
    company_id       TEXT NOT NULL,
    date              TEXT NOT NULL,
    open_price         REAL,
    high_price         REAL,
    low_price          REAL,
    close_price        REAL,
    volume             INTEGER,
    adjusted_close     REAL,
    PRIMARY KEY (company_id, date),
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE financial_ratios (
    company_id                     TEXT NOT NULL,
    year                            TEXT NOT NULL,
    net_profit_margin_pct            REAL,
    operating_profit_margin_pct      REAL,
    return_on_equity_pct             REAL,
    debt_to_equity                   REAL,
    interest_coverage                REAL,
    asset_turnover                   REAL,
    free_cash_flow_cr                REAL,
    capex_cr                         REAL,
    earnings_per_share               REAL,
    book_value_per_share             REAL,
    dividend_payout_ratio_pct        REAL,
    total_debt_cr                    REAL,
    cash_from_operations_cr          REAL,
    PRIMARY KEY (company_id, year),
    FOREIGN KEY (company_id) REFERENCES companies(id)
);

CREATE TABLE peer_groups (
    peer_group_name  TEXT NOT NULL,
    company_id        TEXT NOT NULL,
    benchmark         TEXT,
    PRIMARY KEY (peer_group_name, company_id),
    FOREIGN KEY (company_id) REFERENCES companies(id)
);
CREATE TABLE market_cap (
    company_id             TEXT NOT NULL,
    year                    INTEGER NOT NULL,
    market_cap_crore         REAL,
    enterprise_value_crore   REAL,
    pe_ratio                 REAL,
    pb_ratio                 REAL,
    ev_ebitda                REAL,
    dividend_yield_pct       REAL,
    PRIMARY KEY (company_id, year),
    FOREIGN KEY (company_id) REFERENCES companies(id)
);
"""
# ================================

schema_path = r"C:\Users\bhrra\Desktop\nifty100\db\schema.sql"

os.makedirs(os.path.dirname(schema_path), exist_ok=True)

with open(schema_path, "w", encoding="utf-8") as f:
    f.write(schema_sql)

print("Created:", os.path.exists(schema_path))    


In [ ]:
# Checking tables are populated or not 

conn = sqlite3.connect("nifty100.db")

tables = conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    AND name NOT LIKE 'sqlite_%';
""").fetchall()

print([table[0] for table in tables])
# conn.close()

In [ ]:
# Inseting the processed tables

folder = Path(r"C:\Users\bhrra\Desktop\nifty100\data\processed")
conn = sqlite3.connect("nifty100.db")
for file in folder.iterdir():
    print(file.name)
    print(type(file.name))
    print(file.name[:4])
    print(str(folder/f"{file.name}.csv"))
    folder = Path(r"C:\Users\bhrra\Desktop\nifty100\data\processed")
    print(folder)
    print(folder.joinpath(file.name))
    # break
    df_name = f"{file.name[:-4]}_df"
    # df_name = pd.read_csv()
    # break
    # df.merge(file.name[:4]) = file.name[:4]
    df_name = pd.read_csv(folder.joinpath(file.name))
    print(df_name.head(2))
    df_name.to_sql(
        file.name[:-4],
        conn,
        if_exists="append",
        index=False
    )


### Day 06 & 07 
- Data Quality Manual Review 

- `5 random companies · year coverage · companies with <5 years · fix loader bugs · re-run`

In [ ]:
# Connect
conn = sqlite3.connect("nifty100.db")

queries = {
    "row_counts": """
        SELECT 'companies' AS table_name, COUNT(*) AS rows FROM companies
        UNION ALL SELECT 'profitandloss', COUNT(*) FROM profitandloss
        UNION ALL SELECT 'balancesheet', COUNT(*) FROM balancesheet
        UNION ALL SELECT 'cashflow', COUNT(*) FROM cashflow
        UNION ALL SELECT 'stock_prices', COUNT(*) FROM stock_prices;
    """,

    "company_count": """
        SELECT COUNT(*) AS company_count
        FROM companies;
    """,

    "year_coverage": """
        SELECT
            company_id,
            MIN(year) AS first_year,
            MAX(year) AS last_year,
            COUNT(DISTINCT year) AS years_available
        FROM profitandloss
        GROUP BY company_id
        ORDER BY years_available;
    """,

    "companies_fewer_than_5_years": """
        SELECT
            company_id,
            COUNT(DISTINCT year) AS years_available
        FROM profitandloss
        GROUP BY company_id
        HAVING years_available < 5;
    """,

    "duplicate_company_year": """
        SELECT
            company_id,
            year,
            COUNT(*) AS occurrences
        FROM profitandloss
        GROUP BY company_id, year
        HAVING occurrences > 1;
    """,

    "orphan_profitandloss_company_ids": """
        SELECT p.company_id
        FROM profitandloss p
        LEFT JOIN companies c ON p.company_id = c.id
        WHERE c.id IS NULL;
    """,

    "profitandloss_null_counts": """
        SELECT
            SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END) AS null_sales,
            SUM(CASE WHEN net_profit IS NULL THEN 1 ELSE 0 END) AS null_net_profit,
            SUM(CASE WHEN eps IS NULL THEN 1 ELSE 0 END) AS null_eps
        FROM profitandloss;
    """,

    "sector_distribution": """
        SELECT
            broad_sector,
            COUNT(*) AS company_count
        FROM sectors
        GROUP BY broad_sector
        ORDER BY company_count DESC;
    """,

    "balance_sheet_imbalance": """
        SELECT
            company_id,
            year,
            total_assets,
            total_liabilities,
            ROUND(ABS(total_assets - total_liabilities) * 100.0 / total_assets, 2) AS diff_pct
        FROM balancesheet
        WHERE ABS(total_assets - total_liabilities) * 1.0 / total_assets >= 0.01
        ORDER BY diff_pct DESC;
    """,

    "negative_or_zero_sales": """
        SELECT
            company_id,
            year,
            sales
        FROM profitandloss
        WHERE sales <= 0;
    """
}
for name, query in queries.items():
    print(f"\n{name}")
    df = pd.read_sql_query(query, conn)
    display(df)
conn.close()